In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# EXPERIMENT 09
# SqueezeNet + IG-Guided Diverse Slice Selection
# Varying Number of Selected Slices
# ============================================================

import os
import pandas as pd
import numpy as np

In [3]:
# Paths
VAL_PATH = "/content/drive/MyDrive/ASD_XAI_Project/metadata/val_slices.csv"

IG_PATH = "/content/drive/MyDrive/ASD_XAI_Project/experiment_06_squeezenet_gradcam_ig/ig_complete_ranking.csv"

EXP9_DIR = "/content/drive/MyDrive/ASD_XAI_Project/experiment_09_slice_count_comparison"

os.makedirs(EXP9_DIR, exist_ok=True)

# Number of slices to test
K_VALUES = [5, 8, 10, 15, 20]

print("============================================================")
print("EXPERIMENT 09 — SLICE COUNT COMPARISON")
print("============================================================")
print()
print("K values:", K_VALUES)
print("Experiment folder:", EXP9_DIR)
print()
print("✅ Configuration ready!")

EXPERIMENT 09 — SLICE COUNT COMPARISON

K values: [5, 8, 10, 15, 20]
Experiment folder: /content/drive/MyDrive/ASD_XAI_Project/experiment_09_slice_count_comparison

✅ Configuration ready!


In [4]:
# ============================================================
# CELL 2 — LOAD EXISTING VALIDATION + IG RANKING DATA
# ============================================================

val_df = pd.read_csv(VAL_PATH)
ig_df = pd.read_csv(IG_PATH)

print("============================================================")
print("LOADED EXISTING DATA")
print("============================================================")

print()
print("Validation slices:", len(val_df))
print("Validation subjects:", val_df["SUB_ID"].nunique())

print()
print("IG ranking rows:", len(ig_df))
print("IG ranking subjects:", ig_df["SUB_ID"].nunique())

print()
print("Validation columns:")
print(val_df.columns.tolist())

print()
print("IG ranking columns:")
print(ig_df.columns.tolist())

print()
print("✅ Existing IG scores loaded — no IG recomputation needed.")

LOADED EXISTING DATA

Validation slices: 1054
Validation subjects: 6

IG ranking rows: 1054
IG ranking subjects: 6

Validation columns:
['SUB_ID', 'SLICE_PATH', 'LABEL', 'SPLIT']

IG ranking columns:
['SUB_ID', 'SLICE_PATH', 'LABEL', 'IG_SCORE', 'IG_RANK']

✅ Existing IG scores loaded — no IG recomputation needed.


In [6]:
# ============================================================
# CELL 3 — VERIFY IG RANKING PER SUBJECT
# ============================================================

print("============================================================")
print("IG RANKING SANITY CHECK")
print("============================================================")

summary = (
    ig_df.groupby("SUB_ID")
    .agg(
        TOTAL_SLICES=("SLICE_PATH", "count"),
        BEST_IG_SCORE=("IG_SCORE", "max"),
        WORST_IG_SCORE=("IG_SCORE", "min")
    )
    .reset_index()
)

display(summary)

print()
print("Slices per subject:")
print(
    ig_df.groupby("SUB_ID")
    .size()
    .to_string()
)

print()
print("IG rank range:")
print("Minimum rank:", ig_df["IG_RANK"].min())
print("Maximum rank:", ig_df["IG_RANK"].max())

print()
print("✅ IG ranking verified.")

IG RANKING SANITY CHECK


,SUB_ID,TOTAL_SLICES,BEST_IG_SCORE,WORST_IG_SCORE
0,51460,176,1.291659e-06,7.430265e-11
1,51463,176,7.292763e-07,8.324228e-11
2,51466,176,8.557856e-07,5.225066e-11
3,51475,176,1.132864e-06,5.713138e-11
4,51476,174,1.258858e-06,4.964498e-11
5,51487,176,9.876027e-07,3.553412e-11



Slices per subject:
SUB_ID
51460    176
51463    176
51466    176
51475    176
51476    174
51487    176

IG rank range:
Minimum rank: 1.0
Maximum rank: 176.0

✅ IG ranking verified.


In [7]:
# ============================================================
# CELL 4 — CREATE IG-GUIDED DIVERSE SLICE SELECTIONS
# ============================================================

def select_diverse_slices(subject_df, k):
    """
    Select K high-IG slices while maintaining spatial diversity.
    Slices are ordered by their original IG_RANK, and selection
    is distributed across the slice range.
    """

    subject_df = subject_df.sort_values("IG_RANK").reset_index(drop=True)

    n = len(subject_df)

    # Choose approximately evenly spaced positions
    positions = np.linspace(0, n - 1, k).round().astype(int)

    selected = subject_df.iloc[positions].copy()

    return selected

In [8]:
all_selections = []

for sub_id, group in ig_df.groupby("SUB_ID"):

    for k in K_VALUES:

        selected = select_diverse_slices(group, k)

        selected = selected.copy()
        selected["K"] = k
        selected["SELECTION_METHOD"] = "IG_Diverse"

        all_selections.append(selected)


selection_df = pd.concat(all_selections, ignore_index=True)

print("============================================================")
print("IG-GUIDED DIVERSE SELECTIONS CREATED")
print("============================================================")

print()
print("Total selected rows:", len(selection_df))

print()
print("Selections per K:")
print(
    selection_df.groupby("K")
    .size()
    .to_string()
)

print()
print("Selections per subject and K:")
print(
    selection_df.groupby(["K", "SUB_ID"])
    .size()
    .to_string()
)

print()
print("✅ Diverse slice selections created.")

IG-GUIDED DIVERSE SELECTIONS CREATED

Total selected rows: 348

Selections per K:
K
5      30
8      48
10     60
15     90
20    120

Selections per subject and K:
K   SUB_ID
5   51460      5
    51463      5
    51466      5
    51475      5
    51476      5
    51487      5
8   51460      8
    51463      8
    51466      8
    51475      8
    51476      8
    51487      8
10  51460     10
    51463     10
    51466     10
    51475     10
    51476     10
    51487     10
15  51460     15
    51463     15
    51466     15
    51475     15
    51476     15
    51487     15
20  51460     20
    51463     20
    51466     20
    51475     20
    51476     20
    51487     20

✅ Diverse slice selections created.


In [9]:
# ============================================================
# CELL 5 — LOAD SQUEEZENET MODEL
# ============================================================

import tensorflow as tf

MODEL_PATH = "/content/drive/MyDrive/ASD_XAI_Project/models/squeezenet_baseline.keras"

IMG_SIZE = 224

def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    return image


model = tf.keras.models.load_model(MODEL_PATH)

print("============================================================")
print("SQUEEZENET MODEL LOADED")
print("============================================================")
print()
print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)
print()
print("Preprocessing: resize to 224×224, float32, NO /255 normalization")
print()
print("✅ Model ready.")

SQUEEZENET MODEL LOADED

Input shape : (None, 224, 224, 3)
Output shape: (None, 1)

Preprocessing: resize to 224×224, float32, NO /255 normalization

✅ Model ready.


In [10]:
# ============================================================
# CELL 6 — GENERATE PREDICTIONS FOR ALL SELECTED SLICES
# ============================================================

print("============================================================")
print("GENERATING SQUEEZENET PREDICTIONS")
print("============================================================")

images = []
valid_rows = []

for idx, row in selection_df.iterrows():

    try:
        image = load_image(row["SLICE_PATH"])
        images.append(image)
        valid_rows.append(idx)

    except Exception as e:
        print(f"⚠️ Could not load slice: {row['SLICE_PATH']}")
        print("Error:", e)


# Stack images into one batch
image_batch = tf.stack(images)

print()
print("Images loaded:", image_batch.shape[0])
print("Image tensor shape:", image_batch.shape)

# Predict
predictions = model.predict(
    image_batch,
    verbose=1
).flatten()


# Attach predictions to selection dataframe
predictions_df = selection_df.loc[valid_rows].copy()
predictions_df["PREDICTION_PROBABILITY"] = predictions

print()
print("============================================================")
print("PREDICTIONS COMPLETE")
print("============================================================")

print()
print("Total predictions:", len(predictions_df))

print(
    "Probability range:",
    predictions_df["PREDICTION_PROBABILITY"].min(),
    "to",
    predictions_df["PREDICTION_PROBABILITY"].max()
)

print()
print("Predictions per K:")
print(
    predictions_df.groupby("K")
    .size()
    .to_string()
)

print()
print("✅ All selected slices predicted.")

GENERATING SQUEEZENET PREDICTIONS

Images loaded: 348
Image tensor shape: (348, 224, 224, 3)
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 509ms/step

PREDICTIONS COMPLETE

Total predictions: 348
Probability range: 0.4359564781188965 to 0.5231908559799194

Predictions per K:
K
5      30
8      48
10     60
15     90
20    120

✅ All selected slices predicted.


In [11]:
# ============================================================
# CELL 7 — SUBJECT-LEVEL EVALUATION FOR EACH K
# ============================================================

results = []
subject_results = []

for k in K_VALUES:

    k_df = predictions_df[predictions_df["K"] == k]

    for sub_id, group in k_df.groupby("SUB_ID"):

        true_label = group["LABEL"].iloc[0]

        # Mean aggregation — same aggregation used in Exp 07
        probability = group["PREDICTION_PROBABILITY"].mean()

        predicted_label = "ASD" if probability >= 0.5 else "Control"

        correct = predicted_label == true_label

        subject_results.append({
            "K": k,
            "SUB_ID": sub_id,
            "TRUE_LABEL": true_label,
            "MEAN_PROBABILITY": probability,
            "PREDICTED_LABEL": predicted_label,
            "CORRECT": correct
        })


# Convert to dataframe
subject_results_df = pd.DataFrame(subject_results)


# Calculate accuracy for each K
for k in K_VALUES:

    k_results = subject_results_df[
        subject_results_df["K"] == k
    ]

    correct_subjects = k_results["CORRECT"].sum()
    total_subjects = len(k_results)

    accuracy = correct_subjects / total_subjects

    results.append({
        "K": k,
        "SLICES_PER_SUBJECT": k,
        "TOTAL_SELECTED_SLICES": k * total_subjects,
        "CORRECT_SUBJECTS": int(correct_subjects),
        "TOTAL_SUBJECTS": total_subjects,
        "ACCURACY": accuracy
    })


results_df = pd.DataFrame(results)


print("============================================================")
print("EXPERIMENT 09 — SUBJECT-LEVEL RESULTS")
print("============================================================")

print()
display(results_df)

print()
print("============================================================")
print("SUBJECT-LEVEL PREDICTIONS")
print("============================================================")

display(subject_results_df)

print()
print("✅ Experiment 09 evaluation complete.")

EXPERIMENT 09 — SUBJECT-LEVEL RESULTS



,K,SLICES_PER_SUBJECT,TOTAL_SELECTED_SLICES,CORRECT_SUBJECTS,TOTAL_SUBJECTS,ACCURACY
0,5,5,30,5,6,0.833333
1,8,8,48,4,6,0.666667
2,10,10,60,5,6,0.833333
3,15,15,90,4,6,0.666667
4,20,20,120,5,6,0.833333



SUBJECT-LEVEL PREDICTIONS


,K,SUB_ID,TRUE_LABEL,MEAN_PROBABILITY,PREDICTED_LABEL,CORRECT
0,5,51460,ASD,0.492323,Control,False
1,5,51463,ASD,0.506142,ASD,True
2,5,51466,ASD,0.503523,ASD,True
3,5,51475,Control,0.487664,Control,True
4,5,51476,Control,0.481771,Control,True
5,5,51487,Control,0.497088,Control,True
6,8,51460,ASD,0.493394,Control,False
7,8,51463,ASD,0.506884,ASD,True
8,8,51466,ASD,0.504651,ASD,True
9,8,51475,Control,0.490910,Control,True



✅ Experiment 09 evaluation complete.


In [12]:
# ============================================================
# CELL 8 — SAVE EXPERIMENT 09 RESULTS
# ============================================================

# Save all selected-slice predictions
predictions_df.to_csv(
    f"{EXP9_DIR}/experiment_09_slice_predictions.csv",
    index=False
)

# Save subject-level results
subject_results_df.to_csv(
    f"{EXP9_DIR}/experiment_09_subject_results.csv",
    index=False
)

# Save K comparison
results_df.to_csv(
    f"{EXP9_DIR}/experiment_09_k_comparison.csv",
    index=False
)


# Identify best K values
best_accuracy = results_df["ACCURACY"].max()

best_k_values = results_df.loc[
    results_df["ACCURACY"] == best_accuracy,
    "K"
].tolist()


# Create experiment summary
experiment09_summary = pd.DataFrame([{
    "EXPERIMENT": "Experiment 09",
    "MODEL": "SqueezeNet",
    "XAI_METHOD": "Integrated Gradients",
    "SELECTION_METHOD": "IG-ranked diverse selection",
    "AGGREGATION_METHOD": "Mean",
    "K_VALUES_TESTED": "5, 8, 10, 15, 20",
    "BEST_K_VALUES": ", ".join(map(str, best_k_values)),
    "BEST_ACCURACY": best_accuracy,
    "BEST_CORRECT_SUBJECTS": int(best_accuracy * 6),
    "TOTAL_SUBJECTS": 6
}])

experiment09_summary.to_csv(
    f"{EXP9_DIR}/experiment_09_summary.csv",
    index=False
)


print("============================================================")
print("EXPERIMENT 09 — RESULTS SAVED")
print("============================================================")
print()

display(results_df)

print()
print("Best accuracy:", f"{best_accuracy * 100:.2f}%")
print("Best K values:", best_k_values)
print()

print("Files saved to:")
print(EXP9_DIR)

print()
print("✅ Experiment 09 completed and saved.")

EXPERIMENT 09 — RESULTS SAVED



,K,SLICES_PER_SUBJECT,TOTAL_SELECTED_SLICES,CORRECT_SUBJECTS,TOTAL_SUBJECTS,ACCURACY
0,5,5,30,5,6,0.833333
1,8,8,48,4,6,0.666667
2,10,10,60,5,6,0.833333
3,15,15,90,4,6,0.666667
4,20,20,120,5,6,0.833333



Best accuracy: 83.33%
Best K values: [5, 10, 20]

Files saved to:
/content/drive/MyDrive/ASD_XAI_Project/experiment_09_slice_count_comparison

✅ Experiment 09 completed and saved.
